# 03 — Modèles LSTM & BiLSTM

**Prérequis** : `01_clustering.ipynb` exécuté.

**Données** : `../data/client_data.pkl`, `../data/vocab.pkl`, `../data/label_maps.json`

**Sortie** : `../models/lstm_best.pt`, `../models/bilstm_best.pt`, `../data/results_lstm.pkl`, `../data/results_bilstm.pkl`

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

EMBED_DIM    = 128
HIDDEN_DIM   = 256
NUM_LAYERS   = 2
N_CLASSES    = 8
DROPOUT      = 0.3
BATCH_SIZE   = 64
LR           = 5e-4
N_EPOCHS     = 15
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## Bloc 2 : Chargement des données

In [ ]:
with open(DATA_DIR + 'client_data.pkl', 'rb') as f:
    data = pickle.load(f)
with open(DATA_DIR + 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX = vocab['<PAD>']

print(f'Vocab : {len(vocab):,} tokens')
print(f'Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')
print(f'Classes : {label_names}')

## Bloc 3 — Dataset + DataLoader

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, data):
        self.texts  = [d[0] for d in data]
        self.labels = [d[1] for d in data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded  = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

loaders = {
    split: DataLoader(
        IntentDataset(data[split]),
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        collate_fn=collate_fn
    )
    for split in ('train', 'val', 'test')
}
x, y = next(iter(loaders['train']))
print(f'Batch — textes: {x.shape}, labels: {y.shape}')

## Bloc 4 : Modèles LSTM & BiLSTM (pattern TP6)

In [ ]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, num_layers=2, dropout=0.3):
        super(LSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim,
                                 num_layers=num_layers, batch_first=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded            = self.embedding(x)              # (batch, seq, embed_dim)
        _, (hidden, _)      = self.lstm(embedded)            # hidden : (num_layers, batch, hidden_dim)
        return self.fc(self.dropout(hidden[-1]))              # (batch, output_dim)


class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim,
                                 num_layers=1, batch_first=True, bidirectional=True)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim * 2, output_dim)  # *2 : forward + backward

    def forward(self, x):
        embedded       = self.embedding(x)                          # (batch, seq, embed_dim)
        _, (hidden, _) = self.lstm(embedded)                        # hidden : (2, batch, hidden_dim)
        last_hidden    = torch.cat([hidden[-2], hidden[-1]], dim=1) # (batch, 2*hidden_dim)
        return self.fc(self.dropout(last_hidden))                   # (batch, output_dim)


lstm_model   = LSTM(len(vocab), EMBED_DIM, HIDDEN_DIM, N_CLASSES, NUM_LAYERS, DROPOUT).to(device)
bilstm_model = BiLSTM(len(vocab), EMBED_DIM, HIDDEN_DIM, N_CLASSES, DROPOUT).to(device)

print(f'LSTM   — Paramètres : {sum(p.numel() for p in lstm_model.parameters()):,}')
print(f'BiLSTM — Paramètres : {sum(p.numel() for p in bilstm_model.parameters()):,}')

## Bloc 5 : Fonction d'entraînement réutilisable

In [ ]:
def train_model(model, loaders, n_epochs, lr, model_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses, val_accs = [], [], []
    best_val_acc = 0.0

    for epoch in range(n_epochs):
        # Train
        model.train()
        total_loss = 0.0
        for texts, labels in loaders['train']:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(texts), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        train_losses.append(total_loss / len(loaders['train']))

        # Val
        model.eval()
        val_loss, preds, targets = 0.0, [], []
        with torch.no_grad():
            for texts, labels in loaders['val']:
                texts, labels = texts.to(device), labels.to(device)
                out = model(texts)
                val_loss += criterion(out, labels).item()
                preds.extend(torch.argmax(out, dim=1).cpu().tolist())
                targets.extend(labels.cpu().tolist())
        val_losses.append(val_loss / len(loaders['val']))
        acc = accuracy_score(targets, preds)
        val_accs.append(acc)

        print(f'[{model_name}] Epoch {epoch+1:2d}/{n_epochs} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | val_acc={acc:.4f}', end='')
        if acc > best_val_acc:
            best_val_acc = acc
            torch.save(model.state_dict(), MODEL_DIR + f'{model_name}_best.pt')
            print(' ✓')
        else:
            print()

    return train_losses, val_losses, val_accs, best_val_acc


def evaluate_model(model, model_name, loaders, label_names):
    model.load_state_dict(torch.load(MODEL_DIR + f'{model_name}_best.pt', map_location=device))
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for texts, labels in loaders['test']:
            texts, labels = texts.to(device), labels.to(device)
            out = model(texts)
            preds.extend(torch.argmax(out, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    acc    = accuracy_score(targets, preds)
    report = classification_report(targets, preds, target_names=label_names)
    print(f'{model_name} — Test Accuracy : {acc:.4f}\n{report}')
    return acc, report, preds, targets

## Bloc 6 : Entraînement LSTM

In [ ]:
lstm_train_losses, lstm_val_losses, lstm_val_accs, lstm_best = train_model(
    lstm_model, loaders, N_EPOCHS, LR, 'lstm'
)

## Bloc 7 : Entraînement BiLSTM

In [ ]:
bilstm_train_losses, bilstm_val_losses, bilstm_val_accs, bilstm_best = train_model(
    bilstm_model, loaders, N_EPOCHS, LR, 'bilstm'
)

## Bloc 8 : Courbes comparées LSTM vs BiLSTM

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lstm_val_losses,   label='LSTM val')
axes[0].plot(bilstm_val_losses, label='BiLSTM val')
axes[0].set_title('Val Loss'); axes[0].legend()

axes[1].plot(lstm_val_accs,   label=f'LSTM (best={lstm_best:.4f})')
axes[1].plot(bilstm_val_accs, label=f'BiLSTM (best={bilstm_best:.4f})')
axes[1].set_title('Val Accuracy'); axes[1].legend()

plt.suptitle('LSTM vs BiLSTM — Courbes entraînement')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'lstm_curves.png', dpi=100)
plt.show()

## Bloc 9 : Évaluation sur le test set

In [ ]:
lstm_acc,   lstm_report,   lstm_preds,   lstm_targets   = evaluate_model(lstm_model,   'lstm',   loaders, label_names)
bilstm_acc, bilstm_report, bilstm_preds, bilstm_targets = evaluate_model(bilstm_model, 'bilstm', loaders, label_names)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, preds, targets, name, acc in [
    (axes[0], lstm_preds,   lstm_targets,   'LSTM',   lstm_acc),
    (axes[1], bilstm_preds, bilstm_targets, 'BiLSTM', bilstm_acc),
]:
    cm = confusion_matrix(targets, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(f'{name} (acc={acc:.4f})')
    ax.set_ylabel('Vrai'); ax.set_xlabel('Prédit')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(MODEL_DIR + 'lstm_confusion.png', dpi=100)
plt.show()

## Bloc 10 : Sauvegarde des résultats

In [ ]:
for name, acc, report, preds, targets, train_l, val_l, val_a, best in [
    ('LSTM',   lstm_acc,   lstm_report,   lstm_preds,   lstm_targets,
     lstm_train_losses, lstm_val_losses, lstm_val_accs, lstm_best),
    ('BiLSTM', bilstm_acc, bilstm_report, bilstm_preds, bilstm_targets,
     bilstm_train_losses, bilstm_val_losses, bilstm_val_accs, bilstm_best),
]:
    results = {
        'model': name, 'test_accuracy': acc, 'best_val_acc': best,
        'train_losses': train_l, 'val_losses': val_l, 'val_accs': val_a,
        'predictions': preds, 'targets': targets, 'report': report,
    }
    fname = DATA_DIR + f'results_{name.lower()}.pkl'
    with open(fname, 'wb') as f:
        pickle.dump(results, f)
    print(f'{name} — test_acc={acc:.4f} | sauvegardé → {fname}')